# 08 — Quadrotor Altitude + Roll PD Stabilization

> **Note (council fix, pass 8, Wise):** the controller below is *PD*, not full *PID* — there is no integral term. PD + gravity feedforward gives zero steady-state error to a constant setpoint **only if** the gravity term $mg$ is exact. Any actuator-gain mismatch (battery droop, motor model error) introduces a constant disturbance that PD alone cannot cancel; the I term handles this in production. The label "PID" is retained in the notebook filename for backwards compatibility but the implemented controller is PD.

**Section:** UAV · **Mirrors MATLAB:** *Approximate High-Fidelity UAV Models*

We use a planar (2-D) quadrotor model with state `[z, ż, φ, φ̇]` (altitude, vertical velocity, roll, roll rate) and two control inputs: total **thrust** and **roll torque**.


## Intuition — what's actually going on?

A quadrotor is unstable. Without active control it falls over the moment any disturbance tips it. We need controllers that, given the current attitude and altitude, compute thrust and torque commands to keep it flying.

We use the simplest possible scheme: **cascaded PID**. Two independent control loops:

- **Altitude loop**: thrust ≈ gravity (to hover) plus a correction proportional to "how far am I from the desired altitude" and "how fast am I going up/down".
- **Attitude loop**: torque is proportional to "how tilted am I from upright" and "how fast am I tilting".

The clever part is the **gravity feed-forward**: the controller knows that at hover, thrust must equal weight. So instead of feedback alone catching up, we pre-set the operating point at $T = m g$. The feedback then only has to handle deviations from hover, which keeps the closed loop nicely linear and easy to tune.

The attitude loop is *much faster* than the altitude loop (its eigenvalues are ~42 rad/s vs ~3 rad/s) — that's why the simulation timestep has to be small (0.005 s in the notebook). Bigger timesteps and the attitude loop becomes numerically unstable, even though it's stable on paper. This is a textbook lesson in why simulator timesteps matter.


## Analytical derivation

Derive the model from Newton-Euler, then design PD controllers analytically, then verify the closed loop is stable — only then write code.

### Planar 2-D quadrotor model

State $\mathbf{x} = [z,\ \dot z,\ \phi,\ \dot\phi]^T$ where $z$ is altitude and $\phi$ is roll angle from vertical (positive = tilted right). Inputs: total thrust $T$ along the body's "up" axis, and roll torque $\tau$ about the CoM.

When the body is rolled by $\phi$, its up-axis in world coordinates is

$$\hat{u}_b \;=\; (-\sin\phi,\ \cos\phi).$$

### Newton-Euler

Translation (Newton's 2nd law in world frame, gravity acts in $-\hat{y}$):

$$m\,\ddot{\mathbf r}_{\text{world}} \;=\; T\hat{u}_b \;+\; m\mathbf{g} \;=\; (-T\sin\phi,\ T\cos\phi - mg)$$

**Council note (pass 8, Hovakimyan):** the horizontal channel is *not* truly decoupled — $\ddot x = -T\sin\phi/m$ depends on both thrust (altitude loop) and roll (attitude loop). When the attitude loop wobbles $\phi$, the horizontal channel gets kicked. We don't simulate $x$ here for simplicity; on hardware lateral drift requires an outer position-loop with the same cascaded structure. Focusing on the altitude channel gives

$$\boxed{\;\ddot z \;=\; \frac{T\cos\phi}{m} - g\;}$$

Rotation (scalar in 2-D, since roll is the only rotational DoF):

$$\boxed{\;\ddot\phi \;=\; \frac{\tau}{I}\;}$$

where $I$ is the body's moment of inertia about the roll axis.

### PD control laws

Hover requires $T = mg$ (cancels gravity at $\phi = 0$). We use **gravity-feedforward + PD** on altitude, and pure PD on attitude:

$$T \;=\; m\!\left(g \;+\; K_p^z(z_\text{ref} - z) \;-\; K_d^z\,\dot z\right)$$
$$\tau \;=\; K_p^\phi(\phi_\text{ref} - \phi) \;-\; K_d^\phi\,\dot\phi$$

### Closed-loop analysis (near hover)

Substitute $T$ into the altitude equation and use $\cos\phi \approx 1$ for small angles:

$$\ddot z \;=\; \frac{m\!\left(g + K_p^z(z_\text{ref}-z) - K_d^z \dot z\right)}{m} - g \;=\; K_p^z(z_\text{ref}-z) \;-\; K_d^z\,\dot z$$

That is a standard linear 2nd-order system. With $\tilde z = z - z_\text{ref}$:

$$\ddot{\tilde z} \;+\; K_d^z\,\dot{\tilde z} \;+\; K_p^z\,\tilde z \;=\; 0,\quad \omega_n^z = \sqrt{K_p^z},\quad \zeta^z = \frac{K_d^z}{2\sqrt{K_p^z}}$$

Identically for attitude (with $I$ in the denominator):

$$\ddot\phi \;+\; \frac{K_d^\phi}{I}\,\dot\phi \;+\; \frac{K_p^\phi}{I}\,\phi \;=\; 0,\quad \omega_n^\phi = \sqrt{\tfrac{K_p^\phi}{I}},\quad \zeta^\phi = \frac{K_d^\phi}{2\sqrt{K_p^\phi I}}$$

With our chosen gains ($K_p^z=8$, $K_d^z=4.5$, $K_p^\phi=18$, $K_d^\phi=5.5$, $I=0.01$):

- altitude: $\omega_n^z = 2.83$ rad/s, $\zeta^z = 0.80$ (well-damped)
- attitude: $\omega_n^\phi = 42.4$ rad/s, $\zeta^\phi = 6.48$ (overdamped — sluggish but very robust)

Both loops have all closed-loop poles in the open left-half plane → asymptotically stable around hover.

### Numerical-integration constraint

The attitude loop is **stiff** ($\omega_n^\phi = 42$ rad/s, much faster than altitude). Explicit Euler integration is stable iff $\Delta t \cdot \omega_n^\phi \lesssim 2$. We use $\Delta t = 0.005$ s here ($\Delta t \cdot \omega_n^\phi = 0.21$, safe). The animation in `scripts/build_animations.py` uses $\Delta t = 0.002$ s for margin around disturbances — and we discovered that $\Delta t = 0.01$ pushes the discrete eigenvalues outside the unit disk, blowing the sim up to NaN.

### Compatibility check — math ↔ code

| Derivation step | Code |
|---|---|
| $\ddot z = T\cos\phi/m - g$ | `z_ddot = thrust * np.cos(phi) / m - g` |
| $\ddot\phi = \tau/I$ | `phi_ddot = torque / I` |
| $T = m(g + K_p^z(z_\text{ref}-z) - K_d^z\dot z)$ | `thrust = m * (g + Kp_z*(z_ref - z) - Kd_z*z_dot)` |
| $\tau = K_p^\phi(\phi_\text{ref}-\phi) - K_d^\phi\dot\phi$ | `torque = Kp_phi*(phi_ref - phi) - Kd_phi*phi_dot` |
| $\Delta t < 2/\omega_n^\phi \approx 0.047$ | `dt = 0.005` (well below the bound) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
m = 0.5
I = 0.01

Kp_z, Kd_z = 8.0, 4.5
Kp_phi, Kd_phi = 18.0, 5.5

z_ref = 2.0
phi_ref = 0.0


In [ ]:
x = np.array([0.0, 0.0, 0.30, 0.0])  # start at ground, 0.3 rad roll
dt = 0.005
T = 5.0
N = int(T / dt)
hist = np.zeros((N, 4))
u_hist = np.zeros((N, 2))

for i in range(N):
    z, z_dot, phi, phi_dot = x
    thrust = m * (g + Kp_z * (z_ref - z) - Kd_z * z_dot)
    torque = Kp_phi * (phi_ref - phi) - Kd_phi * phi_dot
    z_ddot = thrust * np.cos(phi) / m - g
    phi_ddot = torque / I
    x = x + dt * np.array([z_dot, z_ddot, phi_dot, phi_ddot])
    hist[i] = x
    u_hist[i] = [thrust, torque]


In [ ]:
t = np.arange(N) * dt
fig, axs = plt.subplots(2, 2, figsize=(11, 6))
axs[0, 0].plot(t, hist[:, 0]); axs[0, 0].axhline(z_ref, color='r', ls='--')
axs[0, 0].set_title('Altitude (m)')
axs[0, 1].plot(t, hist[:, 2]); axs[0, 1].axhline(phi_ref, color='r', ls='--')
axs[0, 1].set_title('Roll (rad)')
axs[1, 0].plot(t, u_hist[:, 0]); axs[1, 0].set_title('Thrust (N)')
axs[1, 1].plot(t, u_hist[:, 1]); axs[1, 1].set_title('Roll torque (N·m)')
for ax in axs.flat:
    ax.set_xlabel('time (s)'); ax.grid()
plt.tight_layout()
plt.show()


## References & rigor notes

**Cascaded loop stability.** Time-scale separation between the fast attitude loop ($\omega_n^\phi \approx 42$ rad/s) and the slow altitude loop ($\omega_n^z \approx 3$ rad/s) — roughly a decade apart — allows analyzing each independently. Singular-perturbation theory (Khalil, 2002) formalizes this: as the ratio of time constants grows, cascaded stability follows from individual loop stability.

**Discrete-time stability bound.** Explicit Euler integration of a 2nd-order system with natural frequency $\omega_n$ is stable iff $\Delta t \cdot \omega_n < 2$. For our attitude loop ($\omega_n = 42$): $\Delta t < 0.047$ s. We use 0.005 s with a 10× margin. Going to 0.01 s pushes discrete eigenvalues past 1 in magnitude and the sim NaNs — we hit exactly this bug in an earlier commit.

**Complexity.** $O(1)$ per control step.

**References.**
- Mahony, R., Kumar, V., & Corke, P. (2012). *Multirotor aerial vehicles: Modeling, estimation, and control of quadrotor*. IEEE Robotics & Automation Magazine, 19(3), 20-32.
- Åström, K. J., & Murray, R. M. (2008). *Feedback Systems: An Introduction for Scientists and Engineers*, Princeton University Press, ch. 10-11.
- Khalil, H. K. (2002). *Nonlinear Systems*, 3rd ed., Prentice Hall, ch. 11 (singular perturbation).
